<a href="https://colab.research.google.com/github/AniTigerSib/AI-Methods/blob/main/AIlab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Реализация распознавания рукописных цифр из набора MNIST

Вариант 3

In [1]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
import torchvision.transforms.v2 as tfs
import torchvision

## Task 1

In [2]:
transform = tfs.ToPILImage()

mnist_train = torchvision.datasets.MNIST(r'mnist', download=True, train=True)
mnist_test = torchvision.datasets.MNIST(r'mnist', download=True, train=False)

dir_out = 'lab_3'
file_format = 'format.json'
train_data = {'dir': "train", 'data': mnist_train}
test_data = {'dir': "test", 'data': mnist_test}

if not os.path.exists(dir_out):
    os.mkdir(dir_out)

    for info in (train_data, test_data):
        os.mkdir(os.path.join(dir_out, info['dir']))

        for i in range(10):
            os.mkdir(os.path.join(dir_out, info['dir'], f"class_{i}"))


for info in (train_data, test_data):
    for i in range(10):
        path = os.path.join(dir_out, info['dir'], f"class_{i}")
        cls = info['data'].data[info['data'].targets == i]

        for n, x in enumerate(cls):
            x = transform(x)
            x.save(os.path.join(path, f"img_{n}.png"), "png")

targets = dict()
for i in range(10):
    targets[f'class_{i}'] = i

fp = open(os.path.join(dir_out, file_format), "w")
json.dump(targets, fp)
fp.close()

## Task 2

In [3]:
class DigitDataset(data.Dataset):
    def __init__(self, path, train=True, transform=None):
        self.path = os.path.join(path, "train" if train else "test")
        self.transform = transform

        with open(os.path.join(path, "format.json"), "r") as fp:
            self.format = json.load(fp)

        self.length = 0 # размер выборки
        self.files = [] # список с информацией о файлах изображений
        # добавить целевые метки классов
        self.targets = torch.eye(10)


        for _dir, _target in self.format.items():
            path = os.path.join(self.path, _dir)
            list_files = os.listdir(path)
            self.length += len(list_files)
            self.files.extend(map(lambda _x: (os.path.join(path, _x), _target), list_files))

    def __getitem__(self, item):
        path_file, target = self.files[item]
        t = self.targets[target]
        img = Image.open(path_file)

        # добавить преобразования
        if self.transform:
          img = self.transform(img).ravel().float() / 255.0

        return img.cuda(), t.cuda()

    def __len__(self):
        return self.length

## Task 3

In [4]:
# создать экземпляры классов DigitDataset и DataLoader
d_train = DigitDataset("lab_3", transform=tfs.ToImage())
train_data = data.DataLoader(d_train, 16, shuffle=True, drop_last=False)

## Task 4

In [5]:
class MyNet(nn.Module):
  def __init__(self, input_dim, num_hidden_layer_1, num_hidden_layer_2, output_dim):
    super().__init__()
    self.hidden_layer_1 = nn.Linear(input_dim, num_hidden_layer_1).cuda()
    self.hidden_layer_2 = nn.Linear(num_hidden_layer_1, num_hidden_layer_2).cuda()
    self.output_layer = nn.Linear(num_hidden_layer_2, output_dim).cuda()

  def forward(self, x): # прямой проход по сети
    x = self.hidden_layer_1(x)
    x = F.tanh(x)
    x = self.hidden_layer_2(x)
    x = F.tanh(x)
    x = self.output_layer(x)
    x = F.tanh(x)
    return x # тензор с выходными значениями

## Task 5

In [6]:
model = MyNet(28 * 28, 32, 16, 10)
print(model)

MyNet(
  (hidden_layer_1): Linear(in_features=784, out_features=32, bias=True)
  (hidden_layer_2): Linear(in_features=32, out_features=16, bias=True)
  (output_layer): Linear(in_features=16, out_features=10, bias=True)
)


In [7]:
lst = list(model.parameters())
print(lst)
nn.utils.parameters_to_vector(model.parameters()).numel()

[Parameter containing:
tensor([[-0.0335, -0.0159, -0.0303,  ..., -0.0090, -0.0244, -0.0313],
        [ 0.0340,  0.0158, -0.0145,  ..., -0.0091,  0.0280,  0.0059],
        [-0.0035,  0.0089,  0.0197,  ...,  0.0187, -0.0197,  0.0238],
        ...,
        [-0.0290, -0.0175, -0.0081,  ..., -0.0174, -0.0339, -0.0208],
        [ 0.0288,  0.0189, -0.0137,  ..., -0.0030,  0.0006,  0.0098],
        [-0.0276, -0.0020,  0.0246,  ...,  0.0150, -0.0134, -0.0127]],
       device='cuda:0', requires_grad=True), Parameter containing:
tensor([-0.0103,  0.0297, -0.0200,  0.0115, -0.0186, -0.0256,  0.0196, -0.0109,
        -0.0120,  0.0141,  0.0189, -0.0168, -0.0040, -0.0342,  0.0234,  0.0070,
         0.0253, -0.0124,  0.0009,  0.0297,  0.0100,  0.0012, -0.0335, -0.0292,
        -0.0002, -0.0337, -0.0315, -0.0016, -0.0270, -0.0199,  0.0122,  0.0007],
       device='cuda:0', requires_grad=True), Parameter containing:
tensor([[ 0.1422,  0.0929,  0.0134, -0.0865, -0.1573, -0.1155, -0.1762,  0.0260,
       

25818

## Task 6

In [8]:
optimizer = optim.Adam(params=model.parameters(), lr=0.005)
loss_func = nn.CrossEntropyLoss()
epoch_amount = 2
losses = []

In [9]:
model.train()

MyNet(
  (hidden_layer_1): Linear(in_features=784, out_features=32, bias=True)
  (hidden_layer_2): Linear(in_features=32, out_features=16, bias=True)
  (output_layer): Linear(in_features=16, out_features=10, bias=True)
)

In [10]:
for epoch in range(epoch_amount):
  for x_data, y_data in train_data:
    y = model(x_data)
    loss = loss_func(y, y_data)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [11]:
d_test = DigitDataset("lab_3", train=False, transform=tfs.ToImage())
test_data = data.DataLoader(d_test, 500, drop_last=False)

In [12]:
Acc = 0
model.eval()

for x_test, y_test in test_data:
  with torch.no_grad():
    pred = model(x_test)
    pred = torch.argmax(pred, dim=1)
    y = torch.argmax(y_test, dim=1)
    Acc += torch.sum(pred == y).item()

Acc /= len(d_test)
print(Acc)

0.9357


## Task 7

In [13]:
lst = list(model.parameters())
print(lst)
nn.utils.parameters_to_vector(model.parameters()).numel()

[Parameter containing:
tensor([[-0.0335, -0.0159, -0.0303,  ..., -0.0090, -0.0244, -0.0313],
        [ 0.0340,  0.0158, -0.0145,  ..., -0.0091,  0.0280,  0.0059],
        [-0.0035,  0.0089,  0.0197,  ...,  0.0187, -0.0197,  0.0238],
        ...,
        [-0.0290, -0.0175, -0.0081,  ..., -0.0174, -0.0339, -0.0208],
        [ 0.0288,  0.0189, -0.0137,  ..., -0.0030,  0.0006,  0.0098],
        [-0.0276, -0.0020,  0.0246,  ...,  0.0150, -0.0134, -0.0127]],
       device='cuda:0', requires_grad=True), Parameter containing:
tensor([-0.2975,  0.2208,  0.3163,  0.3851,  0.5555,  0.6278, -0.1181, -0.3344,
         0.0628,  0.1471, -0.6595,  0.3263,  0.3423,  0.0773, -0.6015, -0.6424,
         0.2477, -0.0987, -0.3105,  0.2205, -0.0497,  0.6473, -0.3695,  0.0250,
         0.7143, -0.2315, -0.0320, -0.0128, -0.2450,  0.0668,  0.4225,  0.0640],
       device='cuda:0', requires_grad=True), Parameter containing:
tensor([[ 5.4643e-01,  4.8804e-01,  2.0184e-02, -5.3523e-01, -4.9210e-01,
         -7.87

25818

## Task 8

In [14]:
stdict = model.state_dict()
torch.save(stdict, 'model_dnn.tar')

In [15]:
new_model = MyNet(28 * 28, 32, 16, 10)
state_model = torch.load('model_dnn.tar', weights_only=True)
new_model.load_state_dict(state_model)

<All keys matched successfully>